# Preparar y publicar el release de datos (docente)

**Cambio Climático · ESPAM MFL** · Herramienta del docente · Versión 1.0 · octubre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/herramientas/reanalisis/preparar_releases.ipynb)

Genera, para cada provincia de Ecuador, los cinco archivos que usan los estudiantes en la **opción B** (`MODO_DATOS = "release"`) y los publica en un release del repositorio:

| Archivo | Contenido |
|---|---|
| `era5_diario_<provincia>.nc` | ERA5 diario, día local 00–24 h, celdas de 0,25° que tocan la provincia |
| `era5_mensual_<provincia>.nc` | Medias mensuales de las diarias (pr en mm/mes) |
| `era5_terreno_<provincia>.nc` | Geopotencial del terreno del modelo (altura = z / 9,80665) |
| `core_diario_<provincia>.nc` | CORe diario, día 06–06 UTC (01:00–01:00 local) |
| `core_mensual_<provincia>.nc` | Medias mensuales de las diarias de CORe |
| `manifiesto.json` | Tamaño, SHA-256, variables y periodo de cada archivo |

**Es el mismo código que usan los estudiantes en la opción A** (`era5.descargar_zona`, `core.meses_diarios`), aplicado a las 24 provincias. La ventaja del release es que los ~64 000 archivos de 6 h de CORe se descargan una sola vez para todo el curso, no una vez por estudiante.

**Duración aproximada (1981–2024):** ERA5 ≈ 450 celdas × 0,5–1 min = 4 a 8 horas; CORe ≈ 30–60 min por año con 4 hilos. Colab gratuito se desconecta a las ~12 h o por inactividad: **todo es reanudable**. Si se corta, vuelva a ejecutar desde la celda 1 y seguirá donde quedó. Los resultados intermedios quedan en su Drive. Para correrlo en un equipo propio, use la versión de terminal: `python -m herramientas.reanalisis.preparar_releases --help`.

© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

## 1. Configuración

In [ ]:
ETIQUETA   = "datos-v1"        # cámbiela (datos-v2, …) si regenera con otro periodo o método: los cuadernos la usan
ANIO_INI, ANIO_FIN = 1981, 2024
PROVINCIAS = "todas"           # o una lista para probar primero: ["Manabí", "Tungurahua"]
FUENTES    = ["era5", "core"]  # puede preparar una sola
HILOS_CORE = 4                 # no suba de 4: es un servidor público
PUBLICAR   = True              # False = solo generar los archivos en Drive
REEMPLAZAR = False             # True = vuelve a subir aunque el archivo ya esté publicado con el mismo tamaño

# Credenciales: en Colab › Secrets, CDSAPI_KEY (su token del CDS) y GITHUB_TOKEN (token del repositorio,
# permiso Contents: Read and write). Si no están, se piden sin mostrarlas.
CDS_TOKEN = ""

## 2. Entorno

In [ ]:
!pip -q install cdsapi eccodes
import os, sys, pathlib, subprocess, datetime
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/crispaliz/cambio-climatico-espam", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
sys.path.insert(0, str(REPO_DIR))
from herramientas import limites, publicar
from herramientas.reanalisis import era5, core, release
from herramientas.reanalisis import preparar_releases as PR
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(".")
import pandas as pd, xarray as xr
SALIDA = BASE / "CC_releases" / ETIQUETA      # lo que se publica
TRABAJO = BASE / "CC_releases" / "_trabajo"   # celdas de ERA5 y años de CORe (reanudable; se puede borrar al final)
for c in (SALIDA, TRABAJO): c.mkdir(parents=True, exist_ok=True)
ANIOS = range(ANIO_INI, ANIO_FIN + 1)
PROVS = PR.lista_provincias(PROVINCIAS)
print(len(PROVS), "provincias:", ", ".join(PROVS))

## 3. Credencial del CDS y plan de trabajo

Cuántas celdas de ERA5 tiene cada provincia. Las celdas de borde compartidas se descargan una sola vez.

In [ ]:
if "era5" in FUENTES:
    print("Credencial CDS:", era5.credenciales(CDS_TOKEN))
plan = pd.DataFrame([{"provincia": p, "celdas_era5": len(limites.celdas(limites.zona(p)["geom"]))} for p in PROVS])
unicas = pd.concat([limites.celdas(limites.zona(p)["geom"])[["lat", "lon"]] for p in PROVS]).drop_duplicates()
display(plan.set_index("provincia").T)
print(f"Celdas únicas de ERA5: {len(unicas)} · ya descargadas: {len(list((TRABAJO / 'celdas_era5').glob('d_*.nc')))}")

## 4. ERA5 por provincia

Una solicitud por celda al conjunto de series horarias por punto; de horario a diario en hora local; malla de la provincia; mensual. El terreno, en una sola solicitud para todo el país.

In [ ]:
ARCHIVOS = []
if "era5" in FUENTES:
    ARCHIVOS += PR.era5_provincias(PROVS, ANIOS, SALIDA, TRABAJO)

## 5. CORe por provincia

Año por año para el recuadro de Ecuador (de cada archivo de 6 h se piden solo los 8 campos), y luego un archivo por provincia con todo el periodo.

In [ ]:
if "core" in FUENTES:
    ARCHIVOS += PR.core_provincias(PROVS, ANIOS, SALIDA, TRABAJO, HILOS_CORE)

## 6. Control antes de publicar

Revise que no haya provincias con celdas vacías, que el periodo sea completo y que los rangos sean físicos: T media de 5 a 30 °C, HR de 40 a 100 %, q de unos 15–20 g/kg en la Costa y menos en la Sierra, presión menor en la Sierra.

In [ ]:
filas = []
for f in sorted(ARCHIVOS):
    with xr.open_dataset(f) as ds:
        fila = {"archivo": f.name, "MB": round(f.stat().st_size / 1e6, 1), "malla": f"{ds.sizes['lat']}×{ds.sizes['lon']}"}
        if "time" in ds.dims:
            fila.update({"desde": str(ds.time.values[0])[:10], "hasta": str(ds.time.values[-1])[:10], "pasos": ds.sizes["time"]})
            for v in ("tmed", "pr", "hr", "q", "ps"):
                if v in ds: fila[v] = round(float(ds[v].mean()), 1)
            fila["vacíos_%"] = round(100 * float(ds["tmed"].isnull().mean()), 1)
        filas.append(fila)
CONTROL = pd.DataFrame(filas).set_index("archivo")
display(CONTROL)
MAN = publicar.manifiesto(ARCHIVOS, SALIDA / "manifiesto.json",
                          {"etiqueta": ETIQUETA, "periodo": [ANIO_INI, ANIO_FIN], "fecha": datetime.date.today().isoformat()})
print(f"Total: {CONTROL['MB'].sum():.0f} MB en {len(ARCHIVOS)} archivos")

## 7. Publicar en GitHub

Crea el release `ETIQUETA` si no existe y sube los archivos. Los que ya estén publicados con el mismo tamaño se mantienen (salvo `REEMPLAZAR = True`). El token de GitHub es **solo del docente**: los estudiantes descargan sin cuenta porque el repositorio es público.

In [ ]:
if PUBLICAR:
    estado = publicar.publicar(ARCHIVOS + [MAN], release.REPO, ETIQUETA, publicar.token(),
                               f"Datos de reanálisis por provincia ({ETIQUETA})",
                               PR.notas(ANIOS, FUENTES, ETIQUETA), REEMPLAZAR)

## 8. Después de publicar

- Si usó una etiqueta distinta de la predeterminada, actualice `ETIQUETA` en `herramientas/reanalisis/release.py` (o pida a los estudiantes `os.environ["CC_DATOS_TAG"] = "datos-v2"` antes del entorno).
- Pruebe la opción B con una provincia y un cantón en el cuaderno común `CR01_descarga_reanalisis`.
- Puede borrar `CC_releases/_trabajo` cuando todo esté publicado. Si lo deja, una nueva versión con más años solo descargará lo que falte en CORe; ERA5 se vuelve a pedir por celda porque el periodo cambia.